# Proyecto EDA con Python: campañas de marketing bancario


## 1. Importación de librerías y configuración

Se utilizan `pandas` y `numpy` para la manipulación de datos, `matplotlib` para las visualizaciones y `pathlib` para trabajar con rutas de archivos de forma ordenada.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGES_DIR = PROJECT_ROOT / "images"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga de datos

El proyecto contiene dos archivos: el CSV principal de campañas y un Excel con tres hojas de información de clientes.

In [ ]:
bank_raw = pd.read_csv(RAW_DIR / "bank-additional.csv")
excel_file = pd.ExcelFile(RAW_DIR / "customer-details.xlsx")

print("Dimensión bank-additional:", bank_raw.shape)
print("Hojas del Excel:", excel_file.sheet_names)
bank_raw.head()

In [ ]:
customer_sheets = []
for sheet_name in excel_file.sheet_names:
    sheet_df = pd.read_excel(excel_file, sheet_name=sheet_name)
    sheet_df["customer_year"] = int(sheet_name)
    customer_sheets.append(sheet_df)

customer_raw = pd.concat(customer_sheets, ignore_index=True)
print("Dimensión customer-details:", customer_raw.shape)
customer_raw.head()

## 3. Exploración inicial

Antes de limpiar los datos, se revisan tipos, valores ausentes, duplicados y estructura general.

In [ ]:
bank_raw.info()

In [ ]:
missing_bank = bank_raw.isna().sum().sort_values(ascending=False).to_frame("missing_values")
missing_bank["missing_pct"] = (missing_bank["missing_values"] / len(bank_raw) * 100).round(2)
missing_bank.head(15)

In [ ]:
print("Duplicados completos en bank:", bank_raw.duplicated().sum())
print("Duplicados por id_ en bank:", bank_raw["id_"].duplicated().sum())
print("Duplicados completos en customer:", customer_raw.duplicated().sum())
print("Duplicados por ID en customer:", customer_raw["ID"].duplicated().sum())

## 4. Limpieza y transformación

Se crean funciones para evitar código repetido. Se normalizan fechas, decimales con coma, columnas categóricas y se eliminan columnas índice innecesarias.

In [ ]:
MESES_ES = {
    "enero": "01", "febrero": "02", "marzo": "03", "abril": "04",
    "mayo": "05", "junio": "06", "julio": "07", "agosto": "08",
    "septiembre": "09", "setiembre": "09", "octubre": "10",
    "noviembre": "11", "diciembre": "12"
}

def parse_fecha_es(valor):
    """Convierte fechas del tipo '2-agosto-2019' a datetime."""
    if pd.isna(valor):
        return pd.NaT
    texto = str(valor).strip().lower()
    partes = texto.split("-")
    if len(partes) == 3:
        dia, mes, anio = partes
        mes = MESES_ES.get(mes, mes)
        return pd.to_datetime(f"{dia}-{mes}-{anio}", dayfirst=True, errors="coerce")
    return pd.to_datetime(texto, dayfirst=True, errors="coerce")


def convertir_decimal_coma(serie):
    """Convierte números almacenados como texto con coma decimal a float."""
    return pd.to_numeric(
        serie.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    )


def limpiar_bank(df):
    datos = df.copy()
    datos = datos.drop(columns=[c for c in datos.columns if c.startswith("Unnamed")], errors="ignore")
    for columna in ["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]:
        datos[columna] = convertir_decimal_coma(datos[columna])
    datos["date"] = datos["date"].apply(parse_fecha_es)
    for columna in ["job", "marital", "education", "contact", "poutcome", "y"]:
        datos[columna] = datos[columna].astype("string").str.strip().str.lower().fillna("unknown")
    for columna in ["default", "housing", "loan"]:
        datos[columna] = datos[columna].map({0.0: "no", 1.0: "yes"}).fillna("unknown")
    return datos


def limpiar_customer(df):
    datos = df.copy()
    datos = datos.drop(columns=[c for c in datos.columns if c.startswith("Unnamed")], errors="ignore")
    datos = datos.rename(columns={"ID": "id_"})
    datos["Dt_Customer"] = pd.to_datetime(datos["Dt_Customer"], errors="coerce")
    datos["total_children"] = datos["Kidhome"] + datos["Teenhome"]
    return datos

In [ ]:
bank = limpiar_bank(bank_raw)
customer = limpiar_customer(customer_raw)

bank["subscribed"] = (bank["y"] == "yes").astype(int)
bank["call_duration_min"] = bank["duration"] / 60
bank["previously_contacted"] = bank["pdays"].ne(999)
bank["days_since_previous_contact"] = bank["pdays"].where(bank["pdays"].ne(999), np.nan)
bank["contact_year_from_date"] = bank["date"].dt.year
bank["contact_month_from_date"] = bank["date"].dt.month
bank["age_group"] = pd.cut(
    bank["age"], bins=[0, 29, 39, 49, 59, 120],
    labels=["<=29", "30-39", "40-49", "50-59", "60+"]
)

bank.head()

### Decisiones sobre valores ausentes

- En variables categóricas se usa la categoría `unknown` para no perder observaciones.
- En variables numéricas como `age`, `euribor3m` o `cons.price.idx` se conservan los valores ausentes. Para gráficos o cálculos concretos se excluyen con `dropna()` cuando corresponde.
- `pdays = 999` no se interpreta como 999 días reales, sino como ausencia de contacto previo. Por eso se crea `previously_contacted` y `days_since_previous_contact`.

In [ ]:
bank.isna().sum().sort_values(ascending=False).head(12)

## 5. Unión de datasets

El CSV y el Excel se unen mediante el identificador común del cliente (`id_`).

In [ ]:
df = bank.merge(customer, on="id_", how="left", indicator=True)
df["income_group"] = pd.qcut(df["Income"], q=4, labels=["bajo", "medio-bajo", "medio-alto", "alto"])

print(df.shape)
df["_merge"].value_counts()

In [ ]:
print("Clientes del Excel no presentes en campañas:", (~customer["id_"].isin(bank["id_"])).sum())
df.head()

## 6. Análisis descriptivo

Se analizan las variables numéricas principales y la distribución de la variable objetivo.

In [ ]:
df[["age", "duration", "campaign", "previous", "Income", "NumWebVisitsMonth", "total_children"]].describe().T

In [ ]:
target_counts = df["y"].value_counts()
target_pct = df["y"].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({"clientes": target_counts, "porcentaje": target_pct})

In [ ]:
df.groupby("y")[["duration", "Income", "NumWebVisitsMonth"]].agg(["count", "mean", "median", "std"]).round(2)

## 7. Análisis por grupos

Se revisan tasas de conversión por profesión, educación, edad, ingresos, método de contacto y resultado anterior.

In [ ]:
def tasa_conversion_por(columna):
    tabla = df.groupby(columna, observed=True)["subscribed"].agg(["count", "mean"])
    tabla["conversion_pct"] = (tabla["mean"] * 100).round(2)
    return tabla.sort_values("conversion_pct", ascending=False)

tasa_conversion_por("job")

In [ ]:
tasa_conversion_por("education")

In [ ]:
tasa_conversion_por("age_group")

In [ ]:
tasa_conversion_por("income_group")

In [ ]:
tasa_conversion_por("poutcome")

In [ ]:
tasa_conversion_por("contact")

## 8. Visualización de datos

Los gráficos se guardan en la carpeta `images` para poder reutilizarlos en el informe y en GitHub.

In [ ]:
def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / nombre, dpi=150, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(7, 4))
conteo = df["y"].value_counts()
plt.bar(conteo.index.astype(str), conteo.values)
plt.title("Distribución de la variable objetivo")
plt.xlabel("Suscripción del depósito")
plt.ylabel("Número de clientes")
guardar_figura("notebook_01_objetivo.png")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["age"].dropna(), bins=30)
plt.title("Distribución de la edad")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
guardar_figura("notebook_02_edad.png")

In [ ]:
job_rate = df.groupby("job")["subscribed"].mean().sort_values(ascending=False) * 100
plt.figure(figsize=(9, 5))
plt.barh(job_rate.index[::-1], job_rate.values[::-1])
plt.title("Tasa de conversión por profesión")
plt.xlabel("Tasa de conversión (%)")
plt.ylabel("Profesión")
guardar_figura("notebook_03_conversion_profesion.png")

In [ ]:
plt.figure(figsize=(7, 4))
df.boxplot(column="duration", by="y")
plt.suptitle("")
plt.title("Duración de llamada según contratación")
plt.xlabel("Suscripción")
plt.ylabel("Duración en segundos")
guardar_figura("notebook_04_boxplot_duracion.png")

In [ ]:
pout_rate = df.groupby("poutcome")["subscribed"].mean().sort_values(ascending=False) * 100
plt.figure(figsize=(7, 4))
plt.bar(pout_rate.index.astype(str), pout_rate.values)
plt.title("Conversión según campaña anterior")
plt.xlabel("Resultado anterior")
plt.ylabel("Tasa de conversión (%)")
guardar_figura("notebook_05_conversion_poutcome.png")

In [ ]:
num_cols = ["age", "duration", "campaign", "pdays", "previous", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed", "Income", "Kidhome", "Teenhome", "NumWebVisitsMonth", "total_children", "subscribed"]
corr = df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, aspect="auto")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Mapa de correlaciones")
guardar_figura("notebook_06_correlaciones.png")

## 9. Correlaciones con la variable objetivo

La correlación no implica causalidad, pero sirve para detectar relaciones lineales relevantes.

In [ ]:
corr_target = corr["subscribed"].drop("subscribed").sort_values(key=lambda x: x.abs(), ascending=False)
corr_target.to_frame("correlacion_con_subscribed")

## 10. Guardado de datos procesados

Se guardan datasets limpios y unidos para que el proyecto sea reproducible.

In [ ]:
bank.to_csv(PROCESSED_DIR / "bank_clean.csv", index=False)
customer.to_csv(PROCESSED_DIR / "customer_details_clean.csv", index=False)
df.to_csv(PROCESSED_DIR / "bank_customer_merged.csv", index=False)

print("Archivos guardados en:", PROCESSED_DIR)

## 11. Conclusiones

1. La tasa general de contratación del depósito es baja, del **11.27%**, por lo que existe un fuerte desbalance entre clientes que contratan y clientes que no contratan.
2. La **duración de la llamada** es la variable más asociada a la contratación: los clientes que suscriben presentan una duración media de **551.62 segundos**, frente a **220.43 segundos** en los que no suscriben.
3. El resultado de campañas anteriores es muy relevante. Cuando `poutcome` es `success`, la tasa de conversión es claramente superior a la del resto de grupos.
4. El contacto `cellular` presenta una conversión superior al contacto `telephone`.
5. Por profesión destacan especialmente `student` y `retired`, mientras que `blue-collar` y `services` muestran tasas de conversión inferiores.
6. Los ingresos y las visitas web no presentan una diferencia fuerte entre contratantes y no contratantes. En este dataset parecen tener menos capacidad explicativa que las variables de campaña.

En resumen, el análisis indica que la entidad debería priorizar clientes con historial positivo de campañas anteriores y mejorar la calidad de las llamadas, en vez de aumentar indiscriminadamente el número de contactos.